### Step 1: Load Dataset and Set Random Seed （10%）

Load the pre-split w8 datasets(w8_laptop_test.csv, w8_laptop_train.csv) from GitHub and set the random seed to 42 using NumPy's random seed function.

https://github.com/Taliahsieh/Machine_Learning_2024/tree/main

In [21]:
import numpy as np # provide array
import pandas as pd
np.random.seed(42)

# Load the pre-split datasets from GitHub
base = "https://raw.githubusercontent.com/Taliahsieh/Machine_Learning_2024/main/"
train = pd.read_csv(base + "w8_laptop_train.csv")
test = pd.read_csv(base + "w8_laptop_test.csv")

print(train.shape, test.shape)

(3379, 18) (597, 18)


###  Step 2: Preprocess the data（10%）
1. Use 'Price' as the label and the rest columns as your features (do it for both training and testing set)（2%）
2. Drop the 'Unnamed: 0' column (i.e a useless feature)（2%）
3. Convert all categorical features into numerical.（6%）

Additionally, print a portion of the X_train data to verify the preprocessing steps.

In [22]:
y_train = train["Price"]
X_train = train.drop(columns=["Price", "Unnamed: 0"])
y_test = test["Price"]
X_test = test.drop(columns=["Price", "Unnamed: 0"])

# extract numerical values from original features
#print(train["RAM"].head())
#print(train["RAM"].str.extract(r"(\d+)").head())
#print(pd.to_numeric(train["RAM"].str.extract(r"(\d+)")[0]).head())

num_cols = ["RAM", "RAM_Expandable", "Ghz", "Display", "SSD", "HDD", "Adapter", "Battery_Life"]

def clean(X):
    X = X.copy()
    for c in num_cols:
        X[c] = pd.to_numeric(X[c].astype(str).str.extract(r"([\d.]+)")[0]).fillna(0)
    X["Gen"] = pd.to_numeric(X["Processor_Name"].str.extract(r"(\d+)(?:th|st|nd|rd) Gen")[0]).fillna(0)
    X["GPU_mem"] = pd.to_numeric(X["GPU"].str.extract(r"(\d+) GB")[0]).fillna(0)
    return X.drop(columns=["Name", "GPU", "RAM_TYPE"])

X_train, X_test = clean(X_train), clean(X_test)

# one-hot encode: 欄位由訓練集決定，測試集對齊訓練集的欄位（沒出現過的類別補 0）
X_train = pd.get_dummies(X_train, dtype=int)
X_test = pd.get_dummies(X_test, dtype=int).reindex(columns=X_train.columns, fill_value=0)

print(X_train.shape, X_test.shape)
X_train.head()



(3379, 190) (597, 190)


,RAM_Expandable,RAM,Ghz,Display,SSD,HDD,Adapter,Battery_Life,Gen,GPU_mem,...,Display_type_LED,GPU_Brand_AMD,GPU_Brand_ATI,GPU_Brand_Apple,GPU_Brand_Intel,GPU_Brand_MediaTek,GPU_Brand_Microsoft,GPU_Brand_NVIDIA,GPU_Brand_Nvidia,GPU_Brand_Qualcomm
0,0.0,16,1.7,15.6,512.0,0.0,0.0,0.0,12.0,6.0,...,0,0,0,0,0,0,0,1,0,0
1,16.0,8,3.3,15.6,512.0,0.0,65.0,8.0,11.0,4.0,...,0,0,0,0,0,0,0,1,0,0
2,16.0,16,2.1,15.6,512.0,0.0,65.0,9.0,0.0,0.0,...,0,1,0,0,0,0,0,0,0,0
3,0.0,16,2.8,15.6,512.0,0.0,65.0,6.0,11.0,0.0,...,0,0,0,0,1,0,0,0,0,0
4,16.0,4,1.2,14.0,256.0,1024.0,65.0,10.0,10.0,0.0,...,0,0,0,0,1,0,0,0,0,0


### Step 3: Feature selection（20%）
This dataset has many features. It is suggested to perform feature selection first. Print the selected features to verify your choices.

In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

# 只用訓練資料，依隨機森林的 feature importance 選出重要度高於平均的特徵
selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), threshold="mean")
selector.fit(X_train, y_train)

selected = X_train.columns[selector.get_support()]
X_train_sel, X_test_sel = X_train[selected], X_test[selected]
print(len(selected), list(selected))

17 ['RAM_Expandable', 'RAM', 'Ghz', 'Display', 'SSD', 'Adapter', 'Battery_Life', 'Gen', 'GPU_mem', 'Brand_Apple', 'Brand_Dell', 'Brand_HP', 'Brand_Lenovo', 'Brand_Microsoft', 'Processor_Name_Intel Core i7 (11th Gen)', 'Processor_Name_Intel Core i9 (12th Gen)', 'GPU_Brand_NVIDIA']


### Step 4: Fine-tune your model using GridSearchCV or cross validation approach on the training data with select features（30%）
Print the best parameter and best CV score. Note: running grid search may take some minutes. Be aware of time.

In [24]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 20],
    "max_features": [0.5, 1.0],
}
grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1), param_grid,
                    cv=5, scoring="neg_root_mean_squared_error")
grid.fit(X_train_sel, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

Best parameters: {'max_depth': 20, 'max_features': 0.5, 'n_estimators': 400}
Best CV RMSE: 22746.61715374167


### Step 5: Calculate the **rmse** on testing data. The lower your RMSE, the higher your score（30%）

Compute the RMSE of your final model on the testing set (do not use the testing set to select features or tune parameters). Your score depends on the RMSE you reach:

| Test RMSE | Score |
|---|---|
| ≥ 50,000, or not computed | 0 / 30 |
| < 50,000 | 15 / 30 |
| < 40,000 | 20 / 30 |
| < 30,000 | 25 / 30 |
| < 26,000 | 30 / 30 |

Print the RMSE so that it is visible in your notebook.

In [25]:
from sklearn.metrics import mean_squared_error

pred = grid.best_estimator_.predict(X_test_sel)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print("Test RMSE:", rmse)

Test RMSE: 24109.07885425784
